In [ ]:
pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")

README.md:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/98.6M [00:00<?, ?B/s]

data/validation1-00000-of-00001.parquet:   0%|          | 0.00/7.92M [00:00<?, ?B/s]

data/validation2-00000-of-00001.parquet:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/6.09M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/91772 [00:00<?, ? examples/s]

Generating validation1 split:   0%|          | 0/5446 [00:00<?, ? examples/s]

Generating validation2 split:   0%|          | 0/2375 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5411 [00:00<?, ? examples/s]

In [ ]:
eli5=eli5.train_test_split(test_size=0.2)

In [ ]:
eli5['train'][0]

{'q_id': '5pwcsq',
 'title': 'What does a virus or bacteria gain from making its host sick?',
 'selftext': '',
 'category': 'Biology',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dcub3qw', 'dcub0bx', 'dcuawcw', 'dcuk7wg', 'dcuedpb'],
  'text': ["Bacteria and viruses don't necessarily want their host sick, or even dead. Complications happen when the virus is in an organism it's not supposed to be in, and that organism isn't equipped to handle the virus. Rats carried the plague and were doing just fine, but when they jumped shipped and infected humans, they wiped out millions in the 14th century. It's the same with the influenza virus, and other viruses. Their goal is to multiply within their host for a while and then spread. Sometimes though, killing your host and exposing its fluids to other organisms is the best way to migrate.",
   "The same thing people gain from turning a forest into a grassland, or a meadow into a city, or by fouling a river by pooping in the river r

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
eli5 = eli5.flatten()

In [ ]:
eli5["train"][0]

{'q_id': '5pwcsq',
 'title': 'What does a virus or bacteria gain from making its host sick?',
 'selftext': '',
 'category': 'Biology',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dcub3qw', 'dcub0bx', 'dcuawcw', 'dcuk7wg', 'dcuedpb'],
 'answers.text': ["Bacteria and viruses don't necessarily want their host sick, or even dead. Complications happen when the virus is in an organism it's not supposed to be in, and that organism isn't equipped to handle the virus. Rats carried the plague and were doing just fine, but when they jumped shipped and infected humans, they wiped out millions in the 14th century. It's the same with the influenza virus, and other viruses. Their goal is to multiply within their host for a while and then spread. Sometimes though, killing your host and exposing its fluids to other organisms is the best way to migrate.",
  "The same thing people gain from turning a forest into a grassland, or a meadow into a city, or by fouling a river by pooping in the river

In [ ]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

In [ ]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2362 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1823 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (8350 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1109 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (3854 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1040 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1804 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1667 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
block_size = 128


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
from transformers import AutoModelForCausalLM,TrainingArguments,Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilbert/distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
training_args= TrainingArguments(
    output_dir="my_awesome_eli5_clm-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.912895,3.790903
2,3.828747,3.780578
3,3.789282,3.778862


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3978, training_loss=3.848653495101967, metrics={'train_runtime': 1009.3477, 'train_samples_per_second': 31.511, 'train_steps_per_second': 3.941, 'total_flos': 1038850556166144.0, 'train_loss': 3.848653495101967, 'epoch': 3.0})

In [ ]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 43.77


In [ ]:
prompt = "Somatic hypermutation allows the immune system to"

In [ ]:
!ls

my_awesome_eli5_clm-model  sample_data


In [ ]:
!ls my_awesome_eli5_clm-model

checkpoint-1000  checkpoint-2000  checkpoint-3000  checkpoint-3978
checkpoint-1500  checkpoint-2500  checkpoint-3500  checkpoint-500


In [ ]:
!ls my_awesome_eli5_clm-model/checkpoint-3978

config.json		optimizer.pt   tokenizer_config.json  training_args.bin
generation_config.json	rng_state.pth  tokenizer.json
model.safetensors	scheduler.pt   trainer_state.json


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "my_awesome_eli5_clm-model/checkpoint-3978"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [ ]:
prompt = """
Question: Why is the sky blue?

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    inputs["input_ids"],
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Question: Why is the sky blue?

Answer:
Blue is a "blue" shape. The blue shape is the result of a change in the shape of the sky. If the blue shape is blue, then the shape of the sky will be blue, and if the blue shape is blue, then the sky will be blue. If the blue shape is blue, then the sky will be blue, and if the blue shape is blue, then the sky will be blue. If the blue shape is blue, then the sky will be blue, and


In [ ]:
prompt="anushka is a nice girl"

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="my_awesome_eli5_clm-model/checkpoint-3978")
generator(prompt)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "anushka is a nice girl with her hair pulled back over a long, long, long, and long hair. That’s what she wants. The blonde hair, the dark hair, the dark brown hair, the black hair, the brown hair, the dark brown hair, and the dark brown hair.Because we don't know when, or how, this is, but it's a bit of a mystery. So, we can't tell. For instance, if you're a woman, it's possible that it's your body detecting when you're in that situation. And it's not. We can't. But it's a very, very, very, very, very close thing to our skin. It's a very, very sensitive spot. It's a very sensitive spot, and you can get into it with your eyes, and your skin can't detect it. And as a result, it might be better to take a look at it in your own skin. So, as a result, it's a very sensitive spot. When you're in that situation, it's not. We can find that you do not have the same amount of blood circulating. But the amount of blood circulating is very, very, very sensitive to your skin. It

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("my_awesome_eli5_clm-model/checkpoint-3978")
inputs = tokenizer(prompt, return_tensors="pt").input_ids

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("my_awesome_eli5_clm-model/checkpoint-3978")
outputs = model.generate(inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [ ]:
tokenizer.batch_decode(outputs, skip_special_tokens=True)

['anushka is a nice girl who gets more attention to the idea of things like the "oh man" and "oh man". Most things I have seen are from childhood to adolescence. When I read a novel or two I would be looking at my aunt and explain how he used to be an important member of society in the past and is the only person in her situation that I know can do that. It is only because her aunt and a friend have been around for decades and I feel a sense of belonging to him. The']